In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

#### Response functions for AIME over (almost) 350 years

In [ ]:
runs = ['wedd','eais','ross','amun','apen']

fig,ax = plt.subplots(1,1,sharex=True,figsize=(5,5))

fname = f'../hpcoutput/larmip_ctrl/scalar_output_ANT_00001.nc'
ds = xr.open_dataset(fname)
ds = ds.sel(time=np.arange(0,351))
IVL_ctrl = ds.ice_volume[0].values-ds.ice_volume.values # m SLE
ds.close()

for r,run in enumerate(runs):

    fname = f'../hpcoutput/larmip_{run}/scalar_output_ANT_00001.nc'
    ds = xr.open_dataset(fname)
    ds = ds.sel(time=np.arange(0,351))
    # Ice volume loss
    IVL = ds.ice_volume[0].values-ds.ice_volume.values # m SLE
    # Ice mass loss relative to control
    IML = (IVL-IVL_ctrl) * 3.618e14*1028*1e-12 # Gt
    # Rate of ice mass loss
    dIdt = np.gradient(IML,ds.time.values) #Gt/y
    # Scale by perturbation to derive response function
    IRF = dIdt / 8 #Gt/m

    ax.plot(ds.time[:-2],IRF[:-2],label=run)
    ds.close()

    # Create new dataset for IRF
    irf_ds = xr.Dataset(
        {'IRF': (['time'], IRF[:-2])},
        coords={'time': ds.time.values[:-2]}
    )
    
    # Inherit attributes from original run
    for key, value in dict(ds.attrs).items():
        irf_ds.attrs[key] = value

    # Add global attributes
    irf_ds.attrs['title'] = f'LARMIP3 Ice mass loss response function from UFEMISM for region {run}'
    irf_ds.attrs['group'] = 'IMAUKNMI'
    irf_ds.attrs['model'] = 'UFEMISM'
    irf_ds.attrs['contact_name'] = 'Erwin Lambert and Constantijn J. (Tijn) Berends'
    irf_ds.attrs['contact_email'] = 'erwin.lambert -at- knmi.nl and c.j.berends -at- uu.nl'
    
    # Variable-specific attributes
    irf_ds['IRF'].attrs['long_name'] = 'Ice mass response function'
    irf_ds['IRF'].attrs['units'] = 'Gt / m'

    ds.close()

    # Write to NetCDF
    outfile = f"response_functions/RF_UFEM_IMAUKNMI_{run}.nc"
    irf_ds.to_netcdf(outfile, encoding={'IRF': {'dtype': 'float64'}})

    with open(f"response_functions/RF_UFEM_IMAUKNMI_{run}","w") as f:
        #Cut off incomplete last 2 years
        for val in IRF[:-2]:
            f.write(f"{val}\n")

ax.set_ylabel('Ice mass response functions (Gt/m)')
ax.legend()

plt.savefig('Response_functions.png')

#### Sea level contributions for LARMIP3 submission

In [ ]:
runs = ['ctrl','wedd','eais','ross','amun','apen']

fig,ax = plt.subplots(1,1,sharey=True,figsize=(5,5))

fname = f'../hpcoutput/larmip_ctrl/scalar_output_ANT_00001.nc'
ds = xr.open_dataset(fname)
ds = ds.sel(time=np.arange(0,201))
# Sea level rise
SLR_ctrl = ds.ice_volume_af[0].values-ds.ice_volume_af.values # m SLE

for r,run in enumerate(runs):

    fname = f'../hpcoutput/larmip_{run}/scalar_output_ANT_00001.nc'
    ds = xr.open_dataset(fname)
    ds = ds.sel(time=np.arange(0,201))
    # Sea level rise
    SLR = ds.ice_volume_af[0].values-ds.ice_volume_af.values # m SLE

    ax.plot(ds.time,SLR-SLR_ctrl,label=run)

    # Create new dataset for SLR
    slr_ds = xr.Dataset(
        {'SLR': (['time'], SLR)},
        coords={'time': ds.time.values}
    )
    
    # Inherit attributes from original run
    for key, value in dict(ds.attrs).items():
        slr_ds.attrs[key] = value

    # Add global attributes
    slr_ds.attrs['title'] = f'LARMIP3 output from UFEMISM for region {run}'
    slr_ds.attrs['group'] = 'IMAUKNMI'
    slr_ds.attrs['model'] = 'UFEMISM'
    slr_ds.attrs['contact_name'] = 'Erwin Lambert and Constantijn J. (Tijn) Berends'
    slr_ds.attrs['contact_email'] = 'erwin.lambert -at- knmi.nl and c.j.berends -at- uu.nl'
    
    # Variable-specific attributes
    slr_ds['SLR'].attrs['long_name'] = 'Sea Level Rise'
    slr_ds['SLR'].attrs['units'] = 'm'

    ds.close()

    # Write to NetCDF
    outfile = f"results/SLR_UFEM_IMAUKNMI_{run}.nc"
    slr_ds.to_netcdf(outfile, encoding={'SLR': {'dtype': 'float64'}})

ax.set_ylabel('Sea level rise w.r.t. ctrl (m)')
ax.set_xlabel('Years')
ax.legend()

plt.savefig('Sea_level_rise.png')